In [2]:
import re
import pandas as pd

In [3]:
file_path = 'data/NCT03860142.ann'

In [5]:
df = pd.read_csv(file_path, sep='^([^\s]*)\s', engine='python', header=None).drop(0, axis=1)
df.columns = ['ID', 'Details']

def extract_text_and_offsets(details):
    parts = details.split('\t')
    entity_info = parts[0].split(' ')
    text_content = parts[-1]
    offsets = ' '.join([part for part in entity_info[1:] if part.isdigit()])
    return text_content, offsets

df['Text'], df['Offsets'] = zip(*df['Details'].apply(extract_text_and_offsets))
t_e_df = df[df['ID'].str.startswith(('T', 'E'))]
text_dict = t_e_df.set_index('ID')['Text'].to_dict()
offset_dict = t_e_df.set_index('ID')['Offsets'].to_dict()

def get_full_text(entity_id):
    if entity_id in text_dict:
        text_content = text_dict[entity_id]
        offsets = offset_dict[entity_id]
        # Wenn die Entität 'E' ist, folge der 'T' Referenz innerhalb
        if entity_id.startswith('E'):
            sub_entity_id = re.search(r'\b(T\d+)\b', text_content)
            if sub_entity_id:
                sub_text, sub_offsets = get_full_text(sub_entity_id.group(1))
                return sub_text, sub_offsets
        return text_content, offsets
    return entity_id, ""


rel_df = df[df['ID'].str.startswith('R')]

rel_pattern = re.compile(r'^(R\d+)\t(And|Or) Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')


relationships = []
for index, row in rel_df.iterrows():
    line = f"{row['ID']}\t{row['Details']}"
    rel_match = rel_pattern.match(line)
    if rel_match:
        rel_id, rel_type, arg1, arg2 = rel_match.groups()
        relationships.append((rel_type, arg1, arg2))


rel_texts = []
for rel_type, arg1, arg2 in relationships:
    arg1_text, arg1_offset = get_full_text(arg1)
    arg2_text, arg2_offset = get_full_text(arg2)
    rel_texts.append({
        "type": rel_type,
        "arg1_text": arg1_text,
        "arg1_offset": arg1_offset,
        "arg2_text": arg2_text,
        "arg2_offset": arg2_offset
    })

rel_texts

[{'type': 'And',
  'arg1_text': 'Children',
  'arg1_offset': '25 33',
  'arg2_text': 'age',
  'arg2_offset': '40 43'},
 {'type': 'And',
  'arg1_text': '≥ 24 months',
  'arg1_offset': '44 55',
  'arg2_text': '≤ 36 months',
  'arg2_offset': '60 71'},
 {'type': 'Or',
  'arg1_text': 'Full-term',
  'arg1_offset': '77 86',
  'arg2_text': 'prematurely',
  'arg2_offset': '90 101'},
 {'type': 'And',
  'arg1_text': 'Children',
  'arg1_offset': '169 177',
  'arg2_text': 'congenital pathologies',
  'arg2_offset': '183 205'},
 {'type': 'Or',
  'arg1_text': 'neurological',
  'arg1_offset': '271 283',
  'arg2_text': 'developmental',
  'arg2_offset': '288 301'},
 {'type': 'And',
  'arg1_text': 'Children',
  'arg1_offset': '257 265',
  'arg2_text': 'pathologies',
  'arg2_offset': '302 313'},
 {'type': 'And',
  'arg1_text': 'Children',
  'arg1_offset': '344 352',
  'arg2_text': 'ENT deformities',
  'arg2_offset': '358 373'}]

In [16]:
from brat_parser import get_entities_relations_attributes_groups
entities, relations, attributes, groups = get_entities_relations_attributes_groups(file_path)

In [14]:
entities    

{'T1': Entity(id='T1', type='Condition', span=((302, 313),), text='pathologies'),
 'T2': Entity(id='T2', type='Condition-Name', span=((302, 313),), text='pathologies'),
 'T3': Entity(id='T3', type='Life-Stage-And-Gender', span=((25, 33),), text='Children'),
 'T4': Entity(id='T4', type='Life-Stage-And-Gender', span=((169, 177),), text='Children'),
 'T5': Entity(id='T5', type='Life-Stage-And-Gender', span=((257, 265),), text='Children'),
 'T6': Entity(id='T6', type='Life-Stage-And-Gender', span=((344, 352),), text='Children'),
 'T9': Entity(id='T9', type='Eq-Operator', span=((44, 45),), text='≥'),
 'T10': Entity(id='T10', type='Eq-Operator', span=((60, 61),), text='≤'),
 'T11': Entity(id='T11', type='Eq-Operator', span=((108, 109),), text='<'),
 'T12': Entity(id='T12', type='Eq-Temporal-Unit', span=((49, 55),), text='months'),
 'T13': Entity(id='T13', type='Eq-Temporal-Unit', span=((65, 71),), text='months'),
 'T14': Entity(id='T14', type='Age', span=((40, 43),), text='age'),
 'T15': Ent

In [7]:
import os
import re
import pandas as pd
import json
def read_ann_files(directory):
    ann_files = [f for f in os.listdir(directory) if f.endswith('.ann')]
    data = {}
    for file in ann_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data[file] = f.read()
    return data

def extract_text_and_offsets(details):
    parts = details.split('\t')
    entity_info = parts[0].split(' ')
    text_content = parts[-1]
    offsets = ' '.join([part for part in entity_info[1:] if part.isdigit()])
    return text_content, offsets

def parse_ann_file(content):
    lines = content.strip().split('\n')
    df = pd.DataFrame([line.split('\t', 1) for line in lines], columns=['ID', 'Details'])
    df['Text'], df['Offsets'] = zip(*df['Details'].apply(extract_text_and_offsets))
    t_e_df = df[df['ID'].str.startswith(('T', 'E'))]
    text_dict = t_e_df.set_index('ID')['Text'].to_dict()
    offset_dict = t_e_df.set_index('ID')['Offsets'].to_dict()
    rel_df = df[df['ID'].str.startswith('R')]
    rel_pattern = re.compile(r'^(R\d+)\t(And|Or) Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')
    relationships = []
    for index, row in rel_df.iterrows():
        line = f"{row['ID']}\t{row['Details']}"
        rel_match = rel_pattern.match(line)
        if rel_match:
            rel_id, rel_type, arg1, arg2 = rel_match.groups()
            relationships.append((rel_type, arg1, arg2))
    return text_dict, offset_dict, relationships

def get_full_text(text_dict, offset_dict, entity_id):
    if entity_id in text_dict:
        text_content = text_dict[entity_id]
        offsets = offset_dict[entity_id]
        if entity_id.startswith('E'):
            sub_entity_id = re.search(r'\b(T\d+)\b', text_content)
            if sub_entity_id:
                sub_text, sub_offsets = get_full_text(text_dict, offset_dict, sub_entity_id.group(1))
                return sub_text, sub_offsets
        return text_content, offsets
    return entity_id, ""

def create_rel_texts(text_dict, offset_dict, relationships):
    rel_texts = []
    for rel_type, arg1, arg2 in relationships:
        arg1_text, arg1_offset = get_full_text(text_dict, offset_dict, arg1)
        arg2_text, arg2_offset = get_full_text(text_dict, offset_dict, arg2)
        rel_texts.append({
            "type": rel_type,
            "arg1_text": arg1_text,
            "arg1_offset": arg1_offset,
            "arg2_text": arg2_text,
            "arg2_offset": arg2_offset
        })
    return rel_texts

def save_intermediate_results(rel_texts, file_prefix, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    file_name = f"{file_prefix}.json"
    file_path = os.path.join(output_directory, file_name)
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(rel_texts, f, indent=4)

directory = 'training'
output_directory = 'training_entitys'
data = read_ann_files(directory)

all_rel_texts = {}
for file, content in data.items():
    text_dict, offset_dict, relationships = parse_ann_file(content)
    rel_texts = create_rel_texts(text_dict, offset_dict, relationships)
    file_prefix = os.path.splitext(file)[0]
    save_intermediate_results(rel_texts, file_prefix, output_directory)
    all_rel_texts[file] = rel_texts

def read_txt_files(directory):
    txt_files = [f for f in os.listdir(directory) if f.endswith('.txt')]
    data = {}
    for file in txt_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data[file] = f.read()
    return data

def insert_relations_to_text(original_text, relations):
    new_text = original_text
    relation_insertions = []

    for rel in relations:
        arg1_end_offset = int(rel['arg1_offset'].split(' ')[-1])
        relation_str = f" [{rel['type'].upper()}] "
        relation_insertions.append((arg1_end_offset + 1, relation_str))

    # Offsets sortieren und Einfügungen vornehmen
    for insert_pos, relation_str in sorted(relation_insertions, reverse=True):
        new_text = new_text[:insert_pos] + relation_str + new_text[insert_pos:]

    return new_text

def write_new_files(new_directory, data, all_rel_texts):
    if not os.path.exists(new_directory):
        os.makedirs(new_directory)
    for file, content in data.items():
        ann_file = file.replace('.txt', '.ann')
        if ann_file in all_rel_texts:
            rel_texts = all_rel_texts[ann_file]
            new_text = insert_relations_to_text(content, rel_texts)
            if new_text is not None:
                file_path = os.path.join(new_directory, file)
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(new_text)

txt_directory = 'training'
new_directory = 'training_parsed_1'
txt_data = read_txt_files(txt_directory)
write_new_files(new_directory, txt_data, all_rel_texts)


In [45]:
# Wörter getrennt
# NCT03925610 